# URW-Depth — Inferență și Vizualizare
Acest notebook rulează inferența pentru **URW-Depth** și **URW-Depth-Weather (Fix4)** pe imagini cu condiții meteorologice adverse și afișează hărțile de disparitate și incertitudine.

In [ ]:
# 1. Clonare repo și instalare dependențe
!git clone https://github.com/marabonea16/TinyDepth-Experiments.git TinyDepth
%cd TinyDepth
!pip install timm==1.0.25 huggingface_hub -q

import torch
print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}, Python OK')
print('mmcv nu este necesar — checkpoint.py folosește torch.load direct')

In [ ]:
# 2. Descărcare weights de pe HuggingFace
from huggingface_hub import hf_hub_download
import os

REPO = 'mara-bonea-16/tinydepth-experiments'
MODELS = {
    'URW-Depth': 'URW-Depth-Calibrated/models/weights_latest',
    'URW-Depth-Weather': 'URW-Depth-S2-Fix4/models/weights_latest',
}
FILES = ['encoder.pth', 'depth.pth']

for model_name, hf_path in MODELS.items():
    local_dir = f'models/{model_name}/weights'
    os.makedirs(local_dir, exist_ok=True)
    for f in FILES:
        dest = f'{local_dir}/{f}'
        if not os.path.exists(dest):
            print(f'Descărcând {model_name}/{f}...')
            path = hf_hub_download(
                repo_id=REPO, repo_type='model',
                filename=f'{hf_path}/{f}',
                local_dir=f'models/{model_name}',
                local_dir_use_symlinks=False
            )
            os.rename(path, dest)
    print(f'  {model_name} OK')

# Backbone TinyViT preantrenat (necesar pentru encoder)
backbone_path = 'networks/tiny_vit_5m_22k_distill_depth.pth'
if not os.path.exists(backbone_path):
    print('Descărcând backbone TinyViT...')
    hf_hub_download(
        repo_id=REPO, repo_type='model',
        filename='Tiny-Depth-b6/models/weights_9/encoder.pth',
        local_dir='/tmp/backbone',
        local_dir_use_symlinks=False
    )
    os.rename('/tmp/backbone/Tiny-Depth-b6/models/weights_9/encoder.pth', backbone_path)
    print('  Backbone OK')

print('Toate weights descărcate!')

In [ ]:
# 3. Import și setup
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image
from torchvision import transforms

sys.path.insert(0, '.')
import networks
from networks.configuration import get_config
from layer import disp_to_depth

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

HEIGHT, WIDTH = 192, 640

In [ ]:
# 4. Funcții de augmentare meteorologică
def apply_snow(img, severity=0.7):
    arr = np.array(img, dtype=np.float32)
    h, w = arr.shape[:2]
    gray = arr.mean(axis=2, keepdims=True)
    arr = arr * (1 - severity * 0.5) + gray * severity * 0.5
    rng = np.random.RandomState(42)
    num_flakes = int(severity * 1800)
    ys, xs = rng.randint(0, h, num_flakes), rng.randint(0, w, num_flakes)
    sizes, alphas = rng.randint(1, 4, num_flakes), rng.uniform(0.7, 1.0, num_flakes)
    for y, x, s, a in zip(ys, xs, sizes, alphas):
        for dy in range(s):
            for dx in range(s):
                arr[min(y+dy,h-1), min(x+dx,w-1)] = arr[min(y+dy,h-1), min(x+dx,w-1)] * (1-a) + 255*a
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def apply_fog(img, severity=0.75):
    arr = np.array(img, dtype=np.float32)
    fog_color = np.array([220, 220, 220], dtype=np.float32)
    h = arr.shape[0]
    gradient = np.linspace(severity, severity * 0.4, h, dtype=np.float32)[:, None, None]
    arr = arr * (1 - gradient) + fog_color * gradient
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def apply_rain(img, severity=0.7):
    arr = np.array(img, dtype=np.float32)
    h, w = arr.shape[:2]
    rng = np.random.RandomState(42)
    for _ in range(int(severity * 1200)):
        x, y = rng.randint(0, w), rng.randint(0, h - 25)
        length, alpha = rng.randint(15, 35), rng.uniform(0.5, 0.85)
        for k in range(length):
            yi, xi = min(y+k, h-1), min(x+k//3, w-1)
            arr[yi, xi] = arr[yi, xi] * (1 - alpha) + 200 * alpha
    arr = arr * (1 - severity * 0.2) + 80 * severity * 0.2
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

print('Funcții augmentare definite.')

In [ ]:
# 5. Încărcare modele
def load_model(weights_dir, use_suppression=False):
    class _Opt:
        img_height = HEIGHT; img_width = WIDTH
        encoder = 'tiny_vit_5m_22k_distill'; scales = [0]
    config = get_config(_Opt())
    enc = networks.build_model(config, img_width=WIDTH, img_height=HEIGHT)
    enc_dict = torch.load(f'{weights_dir}/encoder.pth', map_location=device)
    enc.load_state_dict({k: v for k, v in enc_dict.items() if k in enc.state_dict()})
    enc.to(device).eval()

    num_ch_enc = [64, 64, 128, 160, 320]
    dec = networks.FusionDecoder(num_ch_enc,
                                  use_feature_suppression=use_suppression,
                                  gate_depth_input=False)  # Fix4: pure sigma suppression
    dec.load_state_dict(torch.load(f'{weights_dir}/depth.pth', map_location=device), strict=False)
    dec.to(device).eval()
    return enc, dec

print('Încărcare URW-Depth...')
enc_urw, dec_urw = load_model('models/URW-Depth/weights', use_suppression=False)

print('Încărcare URW-Depth-Weather...')
enc_fix4, dec_fix4 = load_model('models/URW-Depth-Weather/weights', use_suppression=True)

print('Ambele modele încărcate!')

In [ ]:
# 6. Pregătire imagini de intrare
# Încarcă imaginea originală (upload sau URL)
from google.colab import files as colab_files

print('Încarcă imaginea originală (3.jpeg sau orice imagine de drum):')
uploaded = colab_files.upload()
img_path = list(uploaded.keys())[0]

img_orig = Image.open(img_path).convert('RGB')

# Generează variantele cu vreme
images = {
    'Original': img_orig,
    'Ninsoare': apply_snow(img_orig),
    'Ceață': apply_fog(img_orig),
    'Ploaie': apply_rain(img_orig),
}
print(f'Imagini pregătite: {list(images.keys())}')

In [ ]:
# 7. Inferență
to_tensor = transforms.Compose([
    transforms.Resize((HEIGHT, WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def run_inference(encoder, decoder, pil_img):
    inp = to_tensor(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        features = encoder(inp)
        output = decoder(features)
    disp = output[("disp", 0)][0, 0].cpu().numpy()
    sigma = torch.sigmoid(output[("uncert", 0)])[0, 0].cpu().numpy() \
        if ("uncert", 0) in output else np.zeros_like(disp)
    return disp, sigma

results = {}
for name, img in images.items():
    d_urw, s_urw = run_inference(enc_urw, dec_urw, img)
    d_fix4, s_fix4 = run_inference(enc_fix4, dec_fix4, img)
    results[name] = {
        'img': img,
        'URW-Depth': {'disp': d_urw, 'sigma': s_urw},
        'URW-Depth-Weather': {'disp': d_fix4, 'sigma': s_fix4},
    }
    print(f'  {name}: OK')
print('Inferență completă!')

In [ ]:
# 8. Vizualizare — grilă completă
weather_names = list(images.keys())
model_names = ['URW-Depth', 'URW-Depth-Weather']
n_weather = len(weather_names)

# 5 rânduri: imagine, disp URW, sigma URW, disp Fix4, sigma Fix4
row_labels = [
    'Imagine intrare',
    'URW-Depth\nDisparitate',
    'URW-Depth\nIncertitudine (σ)',
    'URW-Depth-Weather\nDisparitate',
    'URW-Depth-Weather\nIncertitudine (σ)',
]
n_rows = len(row_labels)

fig, axes = plt.subplots(n_rows, n_weather, figsize=(5 * n_weather, 3.5 * n_rows))
fig.suptitle('URW-Depth vs URW-Depth-Weather — Disparitate și Incertitudine',
             fontsize=16, fontweight='bold', y=1.01)

for col, wname in enumerate(weather_names):
    res = results[wname]

    # Rând 0: imaginea de intrare
    axes[0, col].imshow(res['img'].resize((WIDTH, HEIGHT)))
    axes[0, col].set_title(wname, fontsize=13, fontweight='bold')
    axes[0, col].axis('off')

    for row_off, mname in enumerate(model_names):
        d = res[mname]['disp']
        s = res[mname]['sigma']
        base_row = 1 + row_off * 2

        # Disparitate
        ax_d = axes[base_row, col]
        im_d = ax_d.imshow(d, cmap='magma', vmin=d.min(), vmax=d.max())
        ax_d.axis('off')
        plt.colorbar(im_d, ax=ax_d, fraction=0.046, pad=0.04)

        # Incertitudine
        ax_s = axes[base_row + 1, col]
        im_s = ax_s.imshow(s, cmap='viridis', vmin=0, vmax=1)
        ax_s.axis('off')
        plt.colorbar(im_s, ax=ax_s, fraction=0.046, pad=0.04)

# Etichete rânduri
for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=11, rotation=0,
                             labelpad=120, va='center')

plt.tight_layout()
plt.savefig('urw_depth_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvat: urw_depth_visualization.png')

In [ ]:
# 9. (Opțional) Descarcă figura
from google.colab import files as colab_files
colab_files.download('urw_depth_visualization.png')